In [1]:
import os
import json
import h5py
import pickle
import shutil
import tempfile
import warnings
import traceback

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)



2026-05-12 03:37:35.082204: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 03:37:35.091982: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778557055.102631      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778557055.105929      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 03:37:35.118865: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices("GPU"))

DATA_DIR = "/kaggle/input/lmsys-chatbot-arena"

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")

df_train = pd.read_csv(train_path)
df_test_data = pd.read_csv(test_path)
df_sample_submission = pd.read_csv(sample_path)

print("Train Shape:", df_train.shape)
print("Test Shape:", df_test_data.shape)
print("Sample Submission Shape:", df_sample_submission.shape)

required_test_cols = {"id", "prompt", "response_a", "response_b"}
required_train_cols = required_test_cols | {
    "winner_model_a",
    "winner_model_b",
    "winner_tie",
}
missing_test = required_test_cols - set(df_test_data.columns)
missing_train = required_train_cols - set(df_train.columns)
if missing_test:
    raise ValueError(f"Missing test columns: {missing_test}")
if missing_train:
    raise ValueError(f"Missing train columns: {missing_train}")



TensorFlow Version: 2.18.0
GPU Available: []


2026-05-12 03:37:40.797021: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Train Shape: (51729, 9)
Test Shape: (5748, 4)
Sample Submission Shape: (5748, 4)


In [3]:
df_test_data["text_concatenated"] = (
    df_test_data["prompt"].fillna("")
    + " [SEP] "
    + df_test_data["response_a"].fillna("")
    + " [SEP] "
    + df_test_data["response_b"].fillna("")
)

df_train["text_concatenated"] = (
    df_train["prompt"].fillna("")
    + " [SEP] "
    + df_train["response_a"].fillna("")
    + " [SEP] "
    + df_train["response_b"].fillna("")
)

y = df_train[["winner_model_a", "winner_model_b", "winner_tie"]].values.astype(
    "float32"
)
print("y shape:", y.shape, "row sums min/max:", y.sum(1).min(), y.sum(1).max())



y shape: (51729, 3) row sums min/max: 1.0 1.0


In [4]:
SEQUENCE_LENGTH = (
    256  # keep modest for runtime; core logic remains sequence+GRU classification
)

print("\n" + "=" * 80)
print("LOADING TOKENIZER (fallback to training-built if external not available)")
print("=" * 80)

external_tokenizer_path = "/kaggle/input/lmsys-hemkesh-10040-gru/tokenizer_gru.pkl"

text_tokenizer = None
if os.path.exists(external_tokenizer_path):
    with open(external_tokenizer_path, "rb") as file_handler:
        text_tokenizer = pickle.load(file_handler)
    print("Loaded external tokenizer:", external_tokenizer_path)
else:
    # Bug fix: external dataset not present; build tokenizer from provided train data
    # This preserves the same general tokenization/sequence core approach.
    print("External tokenizer not found. Building tokenizer from train.csv text...")
    text_tokenizer = keras.preprocessing.text.Tokenizer(
        num_words=50000, oov_token="[OOV]"
    )
    text_tokenizer.fit_on_texts(df_train["text_concatenated"].tolist())

test_sequences = text_tokenizer.texts_to_sequences(
    df_test_data["text_concatenated"].tolist()
)
test_padded_inputs = pad_sequences(
    test_sequences, maxlen=SEQUENCE_LENGTH, padding="post", truncating="post"
).astype("int32")

train_sequences = text_tokenizer.texts_to_sequences(
    df_train["text_concatenated"].tolist()
)
train_padded_inputs = pad_sequences(
    train_sequences, maxlen=SEQUENCE_LENGTH, padding="post", truncating="post"
).astype("int32")

print("X_train shape:", train_padded_inputs.shape)
print("X_test shape:", test_padded_inputs.shape)




LOADING TOKENIZER (fallback to training-built if external not available)
External tokenizer not found. Building tokenizer from train.csv text...
X_train shape: (51729, 256)
X_test shape: (5748, 256)


In [5]:
import tensorflow as tf
from tensorflow import keras


class NonZeroMask(keras.layers.Layer):
    def __init__(self, convert_to_float=True, **kwargs):
        super().__init__(**kwargs)
        self.convert_to_float = convert_to_float

    def call(self, input_tensor):
        if isinstance(input_tensor, (list, tuple)):
            input_tensor = input_tensor[0]
        output_tensor = tf.math.not_equal(input_tensor, 0)
        if self.convert_to_float:
            return tf.cast(output_tensor, tf.float32)
        return output_tensor

    def get_config(self):
        config = super().get_config()
        config.update({"cast_to_float": self.convert_to_float})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)


def load_model_with_custom_config(model_file_path):
    # Kept for compatibility, but will only be used if external models exist.
    with h5py.File(model_file_path, "r") as file_handler:
        architecture_config = json.loads(file_handler.attrs["model_config"])
        for network_layer in architecture_config["config"]["layers"]:
            if network_layer["class_name"] == "NotEqual":
                for graph_node in network_layer.get("inbound_nodes", []):
                    if len(graph_node.get("args", [])) > 1 and isinstance(
                        graph_node["args"][1], int
                    ):
                        graph_node["args"] = [graph_node["args"][0]]

    with tempfile.NamedTemporaryFile(suffix=".h5", delete=False) as temp_file_obj:
        temp_file_path = temp_file_obj.name

    shutil.copy2(model_file_path, temp_file_path)

    with h5py.File(temp_file_path, "r+") as file_handler:
        del file_handler.attrs["model_config"]
        file_handler.attrs["model_config"] = json.dumps(architecture_config)

    loaded_model = keras.models.load_model(
        temp_file_path,
        custom_objects={"NotEqual": NonZeroMask},
        compile=False,
    )
    os.remove(temp_file_path)
    return loaded_model


def build_gru_model(vocab_size: int, seq_len: int):
    # Minimal GRU classifier (sequence -> GRU -> softmax); preserves core GRU approach.
    inputs = keras.Input(shape=(seq_len,), dtype="int32")
    x = keras.layers.Embedding(input_dim=vocab_size, output_dim=128, mask_zero=True)(
        inputs
    )
    x = keras.layers.GRU(128)(x)
    x = keras.layers.Dense(64, activation="relu")(x)
    outputs = keras.layers.Dense(3, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=2e-3),
        loss="categorical_crossentropy",
        metrics=[],
    )
    return model


print("\n" + "=" * 80)
print("LOADING/BUILDING GRU MODELS AND RUNNING INFERENCE")
print("=" * 80)

ensemble_predictions_list = []

# Try loading external fold models if they exist; otherwise train a small in-notebook GRU ensemble.
external_model_dir = "/kaggle/input/lmsys-hemkesh-10040-gru"
external_available = os.path.isdir(external_model_dir)

if external_available:
    k_fold_count = 5
    for current_fold_index in range(k_fold_count):
        input_model_path = (
            f"{external_model_dir}/gru_model_fold_{current_fold_index}.h5"
        )
        print(f"Loading fold {current_fold_index} from {input_model_path}")
        if not os.path.exists(input_model_path):
            print(f"Model file missing: {input_model_path}")
            continue
        try:
            current_gru_model = load_model_with_custom_config(input_model_path)
            fold_output_probs = current_gru_model.predict(
                test_padded_inputs, batch_size=64, verbose=0
            )
            ensemble_predictions_list.append(np.asarray(fold_output_probs))
            print(
                f"Fold {current_fold_index} predictions shape:",
                np.asarray(fold_output_probs).shape,
            )
        except Exception as error_msg:
            print(f"Failed to load fold {current_fold_index}: {repr(error_msg)}")
            traceback.print_exc()

# Fallback: train small GRU(s) quickly if none loaded
if not ensemble_predictions_list:
    print(
        "No external models loaded. Training lightweight GRU ensemble from train.csv..."
    )

    # deterministic shuffle
    n = train_padded_inputs.shape[0]
    idx = np.arange(n)
    rng = np.random.default_rng(SEED)
    rng.shuffle(idx)
    train_padded_inputs = train_padded_inputs[idx]
    y_shuf = y[idx]

    # simple holdout split for stability
    val_size = int(0.05 * n)
    x_val = train_padded_inputs[:val_size]
    y_val = y_shuf[:val_size]
    x_tr = train_padded_inputs[val_size:]
    y_tr = y_shuf[val_size:]

    vocab_size = min(
        getattr(text_tokenizer, "num_words", None)
        or (len(text_tokenizer.word_index) + 1),
        50000,
    )
    vocab_size = int(vocab_size)

    # small 2-model ensemble with different seeds (minimal change; keeps GRU logic)
    for mseed in [SEED, SEED + 1]:
        tf.random.set_seed(mseed)
        np.random.seed(mseed)

        model = build_gru_model(vocab_size=vocab_size, seq_len=SEQUENCE_LENGTH)
        model.fit(
            x_tr,
            y_tr,
            validation_data=(x_val, y_val),
            epochs=2,
            batch_size=128,
            verbose=2,
        )
        preds = model.predict(test_padded_inputs, batch_size=128, verbose=0)
        ensemble_predictions_list.append(np.asarray(preds))
        print("Trained model predictions shape:", np.asarray(preds).shape)

if not ensemble_predictions_list:
    raise RuntimeError(
        "No predictions available: neither external models loaded nor fallback training succeeded."
    )




LOADING/BUILDING GRU MODELS AND RUNNING INFERENCE
No external models loaded. Training lightweight GRU ensemble from train.csv...
Epoch 1/2


2026-05-12 03:37:57.189758: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


384/384 - 59s - 154ms/step - loss: 1.0941 - val_loss: 1.0917
Epoch 2/2
384/384 - 67s - 174ms/step - loss: 0.9886 - val_loss: 1.1573
Trained model predictions shape: (5748, 3)
Epoch 1/2
384/384 - 107s - 278ms/step - loss: 1.0938 - val_loss: 1.0918
Epoch 2/2
384/384 - 106s - 275ms/step - loss: 0.9864 - val_loss: 1.1736
Trained model predictions shape: (5748, 3)


In [6]:
print("\n" + "=" * 80)
print("AVERAGING PREDICTIONS")
print("=" * 80)

# Bug fix: ensure consistent ndarray shapes and avoid scalar mean
pred_stack = np.stack(
    [np.asarray(p) for p in ensemble_predictions_list], axis=0
)  # (k, n, 3)
final_ensemble_probabilities = pred_stack.mean(axis=0)

if final_ensemble_probabilities.ndim != 2 or final_ensemble_probabilities.shape[1] != 3:
    raise ValueError(
        f"Unexpected prediction shape: {final_ensemble_probabilities.shape}"
    )

# Numerical safety for logloss: clip + renormalize
eps = 1e-6
final_ensemble_probabilities = np.clip(final_ensemble_probabilities, eps, 1.0)
final_ensemble_probabilities = (
    final_ensemble_probabilities
    / final_ensemble_probabilities.sum(axis=1, keepdims=True)
)

print("Average predictions shape:", final_ensemble_probabilities.shape)




AVERAGING PREDICTIONS
Average predictions shape: (5748, 3)


In [7]:
print("\n" + "=" * 80)
print("CREATING SUBMISSION")
print("=" * 80)

df_final_submission = df_sample_submission.copy()

# Safety: align to test ids (sample_submission should match, but do not assume ordering mismatches won't happen)
if "id" in df_final_submission.columns and "id" in df_test_data.columns:
    if not df_final_submission["id"].equals(df_test_data["id"]):
        df_final_submission = df_final_submission.set_index("id")
        df_final_submission = df_final_submission.reindex(df_test_data["id"].values)
        df_final_submission = df_final_submission.reset_index()

df_final_submission["winner_model_a"] = final_ensemble_probabilities[:, 0].astype(
    "float32"
)
df_final_submission["winner_model_b"] = final_ensemble_probabilities[:, 1].astype(
    "float32"
)
df_final_submission["winner_tie"] = final_ensemble_probabilities[:, 2].astype("float32")

submission_file_path = "/kaggle/working/submission.csv"
df_final_submission.to_csv(submission_file_path, index=False)

print("Saved:", submission_file_path)
print("Shape:", df_final_submission.shape)
print("Columns:", df_final_submission.columns.tolist())




CREATING SUBMISSION
Saved: /kaggle/working/submission.csv
Shape: (5748, 4)
Columns: ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']


In [8]:
print("\n" + "=" * 80)
print("SUBMISSION PREVIEW")
print("=" * 80)
print(df_final_submission.head(20))




SUBMISSION PREVIEW
            id  winner_model_a  winner_model_b  winner_tie
0   3297560222        0.491335        0.335356    0.173309
1   2556155375        0.357670        0.380856    0.261474
2   1793939629        0.281701        0.349742    0.368557
3   2562993561        0.292306        0.178977    0.528717
4   2111345399        0.353201        0.345964    0.300834
5   2118339565        0.305856        0.186895    0.507249
6   1247276303        0.336886        0.431185    0.231929
7   1825483321        0.609023        0.320532    0.070444
8   1108527003        0.335271        0.367742    0.296987
9   1314503909        0.339867        0.347589    0.312544
10  2154277953        0.376458        0.448888    0.174654
11  3762682310        0.387616        0.304671    0.307713
12  2409640027        0.302020        0.423877    0.274104
13  1050325306        0.335144        0.363069    0.301788
14  1564267187        0.279496        0.415901    0.304603
15  1415251431        0.385911      

In [9]:
print("\n" + "=" * 80)
print("PREDICTION STATISTICS")
print("=" * 80)

a = final_ensemble_probabilities[:, 0]
b = final_ensemble_probabilities[:, 1]
t = final_ensemble_probabilities[:, 2]

print(
    f"Model A - Mean: {a.mean():.6f}, Std: {a.std():.6f}, Min: {a.min():.6f}, Max: {a.max():.6f}"
)
print(
    f"Model B - Mean: {b.mean():.6f}, Std: {b.std():.6f}, Min: {b.min():.6f}, Max: {b.max():.6f}"
)
print(
    f"Tie     - Mean: {t.mean():.6f}, Std: {t.std():.6f}, Min: {t.min():.6f}, Max: {t.max():.6f}"
)




PREDICTION STATISTICS
Model A - Mean: 0.337314, Std: 0.086339, Min: 0.073414, Max: 0.802746
Model B - Mean: 0.356636, Std: 0.100399, Min: 0.036775, Max: 0.802918
Tie     - Mean: 0.306051, Std: 0.110955, Min: 0.026885, Max: 0.873008


In [10]:
print("\n" + "=" * 80)
print("VERIFYING PROBABILITIES SUM TO 1")
print("=" * 80)

probability_row_sums = final_ensemble_probabilities.sum(axis=1)
print("Min sum:", float(probability_row_sums.min()))
print("Max sum:", float(probability_row_sums.max()))
print("Mean sum:", float(probability_row_sums.mean()))




VERIFYING PROBABILITIES SUM TO 1
Min sum: 0.9999998807907104
Max sum: 1.0000001192092896
Mean sum: 1.0


In [11]:
print("\n" + "=" * 80)
print("INFERENCE COMPLETED")
print("=" * 80)
print("Submission file saved:", submission_file_path)


INFERENCE COMPLETED
Submission file saved: /kaggle/working/submission.csv
